# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the `FAIR^2` dataset using the `mlcroissant` library. All data entities—record sets, fields, columns—are referenced by their `@id`.

### Dataset Source
The dataset's Croissant schema is available at:
```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

We use the Croissant schema URL to instantiate the dataset and read its metadata.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL (Croissant schema)
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset from Croissant schema
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # this is a CroissantMetadata object

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
List available record sets, their `@id`s, and associated fields. For each record set, examine its structure and provide an overview of fields (by `@id`) and available columns.

In [ ]:
# Extract and review record set information

if not hasattr(metadata, 'record_sets') or not metadata.record_sets:
    print('No record sets found in metadata.')
else:
    print(f"Found {len(metadata.record_sets)} record set(s):\n")
    for record_set in metadata.record_sets:
        print(f"- RecordSet @id: {record_set.id}")
        print(f"  Name: {record_set.name}")
        if hasattr(record_set, 'fields') and record_set.fields:
            print("  Fields:")
            for field in record_set.fields:
                print(f"    - Field @id: {field.id}, Name: {field.name}, DataType: {getattr(field, 'data_type', 'Unknown')}")
        if hasattr(record_set, 'columns') and record_set.columns:
            print("  Columns:")
            for column in record_set.columns:
                print(f"    - Column @id: {column.id}, Name: {column.name}")
        print()

    # For convenience, collect the list of RecordSet @ids for later use:
    record_set_ids = [rs.id for rs in metadata.record_sets]
    first_record_set_id = record_set_ids[0] if record_set_ids else None

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis, referencing them by their `@id`.

We demonstrate by extracting all available record sets and show data from the first record set, along with its columns.

In [ ]:
# Load all record sets into pandas DataFrames

dataframes = {}
if not hasattr(metadata, 'record_sets') or not metadata.record_sets:
    print('No record sets to extract.')
else:
    for record_set in metadata.record_sets:
        rs_id = record_set.id
        print(f"Extracting records from record set @id: {rs_id}")
        records = list(dataset.records(record_set=rs_id))
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"  Loaded {len(dataframes[rs_id])} records.")

    # Preview columns and head of the first record set
    if dataframes:
        record_set_id = list(dataframes.keys())[0]
        print(f"\nFirst record set (@id: {record_set_id}) columns:")
        print(dataframes[record_set_id].columns.tolist())
        dataframes[record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Perform basic data processing: filtering, normalization, and grouping.

We reference all fields using their `@id`. For this demonstration, you can update `numeric_field_id` and `group_field_id` to valid `@id`s from your record set based on the output above.

In [ ]:
# Specify which record set and fields to use (update as needed based on available fields/columns)
# Example placeholder IDs; please change these to those relevant to your dataset from the overview output above.

record_set_id = first_record_set_id  # Using the first available record set

# Replace these with actual @id from the fields/columns of your record set (see above output)
numeric_field_id = None
group_field_id = None

if record_set_id is not None:
    df = dataframes[record_set_id]
    print(f"Available columns for record set {record_set_id}:")
    print(df.columns.tolist())
    # Attempting to auto-select a numeric column as example
    numeric_candidates = df.select_dtypes(include=['float', 'int']).columns
    if len(numeric_candidates) > 0:
        numeric_field_id = numeric_candidates[0]
        print(f"Using numeric field: {numeric_field_id}")
    # Attempting to auto-select a group-by field (object type)
    group_candidates = df.select_dtypes(include=['object']).columns
    if len(group_candidates) > 0:
        group_field_id = group_candidates[0]
        print(f"Using group field: {group_field_id}")

    # Proceed only if numeric column found
    if numeric_field_id is not None:
        threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # Grouping
        if group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):")
            print(grouped_df.head())
    else:
        print("No numeric field available for EDA.")
else:
    print("No record set loaded for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Ensure columns are referenced by their `@id`.

Here, we provide an example visualization for the selected numeric field. Adjust field IDs as needed for different analyses.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id is not None and numeric_field_id is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id is not None and group_field_id in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load, explore, and begin analyzing the FAIR^2 dataset with the `mlcroissant` library using only `@id` references for all Croissant entities. Data structures can be programmatically explored, sliced, and visualized to support downstream statistical analysis or model development.

Update field and record set references in code cells above as needed, following `@id` outputs from your dataset's metadata.